In [5]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [6]:
# Relative imports
d = os.path.abspath(os.getcwd())
os.chdir("../..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir(d)

### Read (and compact) dataframes

In [7]:
compact = True

In [8]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
data_dir = os.path.join(d, "..", "data")
for file in os.listdir(data_dir):
    fname = os.path.join(data_dir, file)
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(fname)
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(fname, index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")
    
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = os.path.join(d, "trash")
    for f in read:
        os.rename(os.path.join(data_dir, f), os.path.join(trash, f))

dfs = []  # Clear memory
raw_df

Reading compacted_20240905102810.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers,hand_index,state_id
state_id,,,,,,,,,,,,,,,,,,,,
e8471ae6-4685-4fae-903c-bc60a93955dc,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,t,HumanPlayer,call,2,0.3863,0.011589,0,"[3, 0, 0, 0, 0]",172.0,NaN
17ddc421-ed54-4b8e-a543-d8c3f61166f2,e8471ae6-4685-4fae-903c-bc60a93955dc,[],"[96, 91]",0,"[4, 9]","[4, 9]","[True, True]","[False, False]",0,4,t,HumanPlayer,fold,0,0.3863,0.025109,0,"[3, 0, 0, 0, 0]",172.0,NaN
c1a6b9fa-0e54-4a42-be02-9b5dec8cfd3b,None,[],"[92, 96]",0,"[4, 8]","[4, 8]","[False, True]","[False, False]",1,4,t,HumanPlayer,call,4,0.5052,0.030312,0,"[4, 3, 0, 0, 0]",1281.0,NaN
256ec0ab-22bb-443a-8472-9cf8a5a21bbd,c1a6b9fa-0e54-4a42-be02-9b5dec8cfd3b,"[11, 29, 31]","[88, 96]",0,"[0, 0]","[8, 8]","[False, True]","[False, False]",1,4,t,HumanPlayer,raise,8,0.6115,0.048920,1,"[3, 11, 5, 4, 0]",1281.0,NaN
b3b1a507-e05d-402a-b8bf-24586d305db0,256ec0ab-22bb-443a-8472-9cf8a5a21bbd,"[11, 29, 31]","[80, 66]",0,"[8, 30]","[16, 38]","[True, True]","[False, False]",1,4,t,HumanPlayer,call,22,0.6115,0.165105,1,"[3, 11, 5, 4, 0]",1281.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56832d49-3323-4a3c-b61a-d78f1cc32fad,3ca7d536-17db-4180-acb0-f621b7e3c3ee,[],"[88, 100]",0,"[4, 8]","[4, 8]","[True, True]","[False, False]",0,4,Arin,HumanPlayer,raise,10,0.5697,0.034182,0,"[11, 6, 0, 0, 0]",1153.0,NaN
6573af7b-2d08-4b20-a4c1-2ec5cf5c6c4e,56832d49-3323-4a3c-b61a-d78f1cc32fad,"[27, 28, 1]","[78, 94]",0,"[0, 0]","[14, 14]","[False, False]","[False, False]",0,4,Arin,HumanPlayer,raise,30,0.5338,0.074732,1,"[1, 11, 6, 2, 0]",1153.0,NaN
e45b4fdd-5017-45b8-b839-87b08f5c264d,6573af7b-2d08-4b20-a4c1-2ec5cf5c6c4e,"[27, 28, 1]","[48, 54]",0,"[30, 40]","[44, 54]","[True, True]","[False, False]",0,4,Arin,HumanPlayer,raise,48,0.5338,0.261562,1,"[1, 11, 6, 2, 0]",1153.0,NaN


In [5]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
hand_index           float64
state_id             float64
dtype: object

In [6]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [7]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name,n_players
e8471ae6-4685-4fae-903c-bc60a93955dc,e764f112-4567-458c-99c7-907b54b5d6da,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.3863,0.011589,preflop,t,2
17ddc421-ed54-4b8e-a543-d8c3f61166f2,e764f112-4567-458c-99c7-907b54b5d6da,0,0,0,0,0,2,0,0,0,...,0,0,fold,0,0,0.3863,0.025109,preflop,t,2
c1a6b9fa-0e54-4a42-be02-9b5dec8cfd3b,a22d2b4e-cb7e-414c-9ba1-04dc128c6936,0,0,0,0,0,0,0,0,0,...,0,0,call,4,0,0.5052,0.030312,preflop,t,2
256ec0ab-22bb-443a-8472-9cf8a5a21bbd,a22d2b4e-cb7e-414c-9ba1-04dc128c6936,0,0,0,0,0,4,0,0,0,...,0,0,raise,8,1,0.6115,0.048920,flop,t,2
b3b1a507-e05d-402a-b8bf-24586d305db0,a22d2b4e-cb7e-414c-9ba1-04dc128c6936,0,8,0,0,0,4,0,0,0,...,0,0,call,22,1,0.6115,0.165105,flop,t,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56832d49-3323-4a3c-b61a-d78f1cc32fad,92de2f5b-2d33-4d08-8818-40b5b0cf3b00,0,0,0,0,0,2,0,0,0,...,0,0,raise,10,0,0.5697,0.034182,preflop,Arin,2
6573af7b-2d08-4b20-a4c1-2ec5cf5c6c4e,92de2f5b-2d33-4d08-8818-40b5b0cf3b00,10,0,0,0,0,2,0,0,0,...,0,0,raise,30,0,0.5338,0.074732,flop,Arin,2
e45b4fdd-5017-45b8-b839-87b08f5c264d,92de2f5b-2d33-4d08-8818-40b5b0cf3b00,10,30,0,0,0,2,0,0,0,...,0,0,raise,48,0,0.5338,0.261562,flop,Arin,2
d1170aa4-3098-4da3-aa8f-ebaef329d236,92de2f5b-2d33-4d08-8818-40b5b0cf3b00,10,78,0,0,0,2,0,0,0,...,0,0,check,0,0,0.4246,0.390632,turn,Arin,2


In [8]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [10]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [11]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name,n_players
e8471ae6-4685-4fae-903c-bc60a93955dc,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,2,preflop,t,2
17ddc421-ed54-4b8e-a543-d8c3f61166f2,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,fold,0,preflop,t,2
c1a6b9fa-0e54-4a42-be02-9b5dec8cfd3b,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,call,4,preflop,t,2
256ec0ab-22bb-443a-8472-9cf8a5a21bbd,0,0,0,0,0,4,0,0,0,0,...,0,1,0,0,0,raise,8,flop,t,2
b3b1a507-e05d-402a-b8bf-24586d305db0,0,8,0,0,0,4,0,0,0,0,...,0,1,0,0,0,call,22,flop,t,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56832d49-3323-4a3c-b61a-d78f1cc32fad,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,raise,10,preflop,Arin,2
6573af7b-2d08-4b20-a4c1-2ec5cf5c6c4e,10,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,raise,30,flop,Arin,2
e45b4fdd-5017-45b8-b839-87b08f5c264d,10,30,0,0,0,2,0,0,0,0,...,0,0,0,0,0,raise,48,flop,Arin,2
d1170aa4-3098-4da3-aa8f-ebaef329d236,10,78,0,0,0,2,0,0,0,0,...,0,0,0,0,0,check,0,turn,Arin,2


In [12]:
y

e8471ae6-4685-4fae-903c-bc60a93955dc    0
17ddc421-ed54-4b8e-a543-d8c3f61166f2    0
c1a6b9fa-0e54-4a42-be02-9b5dec8cfd3b    0
256ec0ab-22bb-443a-8472-9cf8a5a21bbd    1
b3b1a507-e05d-402a-b8bf-24586d305db0    1
                                       ..
56832d49-3323-4a3c-b61a-d78f1cc32fad    0
6573af7b-2d08-4b20-a4c1-2ec5cf5c6c4e    0
e45b4fdd-5017-45b8-b839-87b08f5c264d    0
d1170aa4-3098-4da3-aa8f-ebaef329d236    0
52c554f1-fb2c-41a6-ad56-de73fe010d10    0
Name: excess_rank, Length: 4652, dtype: int64

In [13]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [14]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (3709, 35)
Test shape: (943, 35)


In [15]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7762460233297985


In [16]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.7762460233297985
Mean goodness:  0.6954504967923356


,0,1,2,3,4,5,true,pred,correct,goodness
e33f3e8d-d258-42de-8625-24f410e58a7d,0.931777,0.067457,0.000297,0.000134,0.000171,0.000164,0,0,True,0.931777
8d7e18f1-48c8-4402-b16e-78095f8ea352,0.910809,0.088418,0.000435,0.000104,0.000085,0.000148,0,0,True,0.910809
9c97d8c9-bd51-4af4-80ad-c0230c92fb3d,0.616196,0.351000,0.022605,0.006931,0.001114,0.002154,0,0,True,0.616196
ddb872bb-e2cd-4373-bb87-6e56626c4caf,0.955034,0.039234,0.001466,0.003005,0.000254,0.001008,0,0,True,0.955034
74cd2a13-c711-43b8-93ec-b326e837a9f7,0.931777,0.067457,0.000297,0.000134,0.000171,0.000164,0,0,True,0.931777
...,...,...,...,...,...,...,...,...,...,...
865f17b9-417b-4852-bb56-4a743315700d,0.961561,0.037754,0.000067,0.000402,0.000067,0.000148,0,0,True,0.961561
b47ba235-a1c6-4b3a-bfc4-5943260568d3,0.931777,0.067457,0.000297,0.000134,0.000171,0.000164,0,0,True,0.931777
d9bccade-81e4-4d38-a9ee-ed3a8669b0f7,0.645682,0.336717,0.008899,0.004916,0.001801,0.001986,0,0,True,0.645682
4b6af43a-066f-45e8-8e88-88a5f90d2dde,0.647088,0.301092,0.009836,0.023738,0.012582,0.005665,0,0,True,0.647088


In [17]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

211 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
9c5d0789-08ba-400f-9e69-59b707e40c54,0.125042,0.695000,0.081972,0.028953,0.050274,0.018761,4,1,False,0.050274
b0874bf3-b9c2-4e92-a437-c36f24db6b7b,0.558425,0.414002,0.006997,0.002084,0.003140,0.015352,1,0,False,0.414002
53b77167-a503-4ae1-9ec7-c80d8421a03e,0.088201,0.752706,0.127533,0.008141,0.007968,0.015451,0,1,False,0.088201
56786439-1770-4086-8a62-fecbd6f9ecf9,0.277779,0.595619,0.015348,0.007351,0.022317,0.081586,0,1,False,0.277779
e3f3715a-deff-47c9-a80e-4ed64e6201bd,0.125279,0.780401,0.074472,0.006020,0.007225,0.006603,0,1,False,0.125279
...,...,...,...,...,...,...,...,...,...,...
810a05eb-5e27-4212-8a98-8b8a1091cf94,0.504188,0.445059,0.018472,0.001101,0.009760,0.021421,1,0,False,0.445059
336eb3d6-af0a-415c-a9f7-63538dce917a,0.385214,0.548541,0.025422,0.000260,0.008359,0.032205,2,1,False,0.025422
526d46df-0537-4c9c-9f45-fb11f9a50f58,0.799044,0.189256,0.002781,0.002010,0.002302,0.004607,1,0,False,0.189256
350e3e75-9d95-4bea-8123-960feab012e6,0.734507,0.238335,0.006218,0.003549,0.003314,0.014077,1,0,False,0.238335


### Attempt to infer card probabilities from rank and table

In [18]:
from cpp_poker.cpp_poker import Hand, CardCollection, HandRank, CheatSheet

In [19]:
hand_ranks_per_state = []
table_ranks = []
for i, (state_id, row) in enumerate(prob_df.iterrows()):
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    table_rank = table_cards.rank_hand().get_rank()
    excess_hand_ranks_row = table_cards.rank_all_hands()
    hand_ranks_per_state.append(
        [rank.get_rank() - table_rank for rank in excess_hand_ranks_row]
    )
    table_ranks.append(table_rank)

table_ranks_df = pd.DataFrame(table_ranks, index=prob_df.index, columns=["table_rank"])
hand_ranks_df = pd.DataFrame(hand_ranks_per_state, index=prob_df.index)
hand_ranks_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
e33f3e8d-d258-42de-8625-24f410e58a7d,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8d7e18f1-48c8-4402-b16e-78095f8ea352,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9c97d8c9-bd51-4af4-80ad-c0230c92fb3d,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,1,1
ddb872bb-e2cd-4373-bb87-6e56626c4caf,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,1,1
74cd2a13-c711-43b8-93ec-b326e837a9f7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
865f17b9-417b-4852-bb56-4a743315700d,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
b47ba235-a1c6-4b3a-bfc4-5943260568d3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
d9bccade-81e4-4d38-a9ee-ed3a8669b0f7,0,0,0,0,0,0,0,0,2,0,...,0,0,1,0,0,1,0,1,0,1
4b6af43a-066f-45e8-8e88-88a5f90d2dde,0,0,0,0,0,0,0,0,4,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
# Get the values of the column indices from hand_ranks_df
column_indices = hand_ranks_df.values

# Use np.arange to create an array of row indices
row_indices = np.arange(hand_ranks_df.shape[0])[:, None]

# Use the row and column indices to index into the DataFrame values
hand_probs_df = pd.DataFrame(
    prob_df.values[row_indices, column_indices],
    index=hand_ranks_df.index,
    columns=hand_ranks_df.columns,
)

# Normalize the hand probabilities to sum to 1 for each row
hand_probs_df = hand_probs_df.div(hand_probs_df.sum(axis=1), axis=0)
hand_probs_df

,0,1,2,3,4,5,6,7,8,9,...,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325
e33f3e8d-d258-42de-8625-24f410e58a7d,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,...,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798
8d7e18f1-48c8-4402-b16e-78095f8ea352,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,...,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796,0.000796
9c97d8c9-bd51-4af4-80ad-c0230c92fb3d,0.00088,0.00088,0.00088,0.00088,0.00088,0.00088,0.00088,0.00088,0.00088,0.00088,...,0.00088,0.00088,0.00088,0.000501,0.00088,0.00088,0.000501,0.00088,0.000501,0.000501
ddb872bb-e2cd-4373-bb87-6e56626c4caf,0.000977,0.000977,0.000977,0.000977,0.000977,0.000977,0.000977,0.000977,0.000977,0.000977,...,0.000977,0.000977,0.000977,0.00004,0.000977,0.000977,0.00004,0.000977,0.00004,0.00004
74cd2a13-c711-43b8-93ec-b326e837a9f7,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,...,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
865f17b9-417b-4852-bb56-4a743315700d,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,...,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799,0.000799
b47ba235-a1c6-4b3a-bfc4-5943260568d3,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,...,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798,0.000798
d9bccade-81e4-4d38-a9ee-ed3a8669b0f7,0.000889,0.000889,0.000889,0.000889,0.000889,0.000889,0.000889,0.000889,0.000012,0.000889,...,0.000889,0.000889,0.000464,0.000889,0.000889,0.000464,0.000889,0.000464,0.000889,0.000464
4b6af43a-066f-45e8-8e88-88a5f90d2dde,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000017,0.000884,...,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884,0.000884


In [21]:
# Validate that rows sum to 1
hand_probs_df.sum(axis=1)

e33f3e8d-d258-42de-8625-24f410e58a7d    1.0
8d7e18f1-48c8-4402-b16e-78095f8ea352    1.0
9c97d8c9-bd51-4af4-80ad-c0230c92fb3d    1.0
ddb872bb-e2cd-4373-bb87-6e56626c4caf    1.0
74cd2a13-c711-43b8-93ec-b326e837a9f7    1.0
                                       ... 
865f17b9-417b-4852-bb56-4a743315700d    1.0
b47ba235-a1c6-4b3a-bfc4-5943260568d3    1.0
d9bccade-81e4-4d38-a9ee-ed3a8669b0f7    1.0
4b6af43a-066f-45e8-8e88-88a5f90d2dde    1.0
7b98f866-1ee2-4ae1-bee0-0753bde38384    1.0
Length: 943, dtype: object

In [22]:
# Look at the max probabilites for different hands
mask = prob_df["pred"] == 2
max_probs = hand_probs_df[mask].max(axis=1)
min_probs = hand_probs_df[mask].min(axis=1)
max_prob_hands = hand_probs_df[mask].idxmax(axis=1)
min_prob_hands = hand_probs_df[mask].idxmin(axis=1)
mean_probs = hand_probs_df[mask].mean(axis=1)
sample_prob_df = pd.DataFrame(
    {
        "max_prob": max_probs,
        "max_prob_hand": max_prob_hands,
        "min_prob": min_probs,
        "min_prob_hand": min_prob_hands,
        "mean_prob": mean_probs,
    }
)
sample_prob_df["max_prob_hand"] = sample_prob_df["max_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["min_prob_hand"] = sample_prob_df["min_prob_hand"].apply(lambda x: Hand(x).str())
sample_prob_df["table"] = raw_df.loc[sample_prob_df.index, "public_cards"].apply(
    lambda x: CardCollection(x).str()
)
sample_prob_df["predicted_excess_rank"] = prob_df.loc[sample_prob_df.index, "pred"]
sample_prob_df["pred_rank"] = (
    sample_prob_df["predicted_excess_rank"]
    + table_ranks_df.loc[sample_prob_df.index, "table_rank"]
)
sample_prob_df["pred_rank_str"] = sample_prob_df["pred_rank"].apply(
    lambda x: HandRank(x, []).get_rank_name()
)
sample_prob_df = sample_prob_df.drop(["predicted_excess_rank"], axis=1)
sample_prob_df

,max_prob,max_prob_hand,min_prob,min_prob_hand,mean_prob,table,pred_rank,pred_rank_str
6162604912,0.001441,♥ 6 ♥ 7,0.000014,♥ 6 ♣ 6,0.000754,♥ Q ♦ 6 ♦ J ♠ 7 ♠ 10,2,Two Pairs
6127537024,0.022439,♣ 4 ♠ 4,0.00001,♥ 2 ♣ 2,0.000754,♥ 4 ♦ 2 ♦ 4 ♦ 8 ♣ 9,3,Three of a Kind


In [24]:
# Deepdive into a specific row
state_id = "6162604912"
row = sample_prob_df.loc[state_id]
most_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=False).head(5)
least_likely_hands = hand_probs_df.loc[state_id].sort_values(ascending=True).head(5)
print("Table cards:", row["table"])
print("Predicted excess rank:", prob_df.loc[state_id, "pred"])
print("Actual excess rank", df.loc[state_id, "excess_rank"])
print("Most and least likely hands:")
for hand, prob in most_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")
print("...")
for hand, prob in least_likely_hands.items():
    print(f"{Hand(hand).str()}: {prob*100:.2f}%")

Table cards: ♥ Q ♦ 6 ♦ J ♠ 7 ♠ 10 
Predicted excess rank: 2
Actual excess rank 1
Most and least likely hands:
♥ J ♠ 6 : 0.14%
♦ 7 ♦ Q : 0.14%
♥ 7 ♣ Q : 0.14%
♥ 7 ♣ J : 0.14%
♥ 7 ♣ 10 : 0.14%
...
♥ 6 ♣ 6 : 0.00%
♥ 10 ♣ 10 : 0.00%
♥ 7 ♦ 7 : 0.00%
♣ J ♠ J : 0.00%
♦ Q ♣ Q : 0.00%


### Attempt to infer winning probabilities from hidden state probabilities

In [25]:
hand_winning_probs = []
for i, (state_id, row) in enumerate(hand_probs_df.iterrows()):
    print(f"Processing row {i}", flush=True)
    raw_row = raw_df.loc[state_id]
    table_cards = CardCollection(raw_row["public_cards"])
    processed_row = df.loc[state_id]
    n_players = processed_row["n_players"]
    hand_winning_probs.append(CheatSheet.get_all_winning_probabilities(table_cards, n_players, 1000))

hand_winning_probs_df = pd.DataFrame(hand_winning_probs, index=hand_probs_df.index)
hand_winning_probs_df

Processing row 0
Cache is not loaded, loading cache
Setting up signal handlers
Processing row 1
Processing row 2
Processing row 3
Processing row 4
Processing row 5
Processing row 6


: 